# Do the tuned supervised-only baselines transfer to new sites? (Tables 17 and 18)

`baseline_tuned.ipynb` showed that a tuned fine-tune on the 70 labels matches probe + distillation in distribution.
The models were not saved there, so this notebook **retrains the same 20 fine-tunes** (same label draws, same seeds,
same settings; GPU training is not bit-identical, so our-test AUC is compared with the first run) and scores them,
**unchanged**, on:

* the field photographs: held-out half (1,248, as in Table 17) and all 2,500, whole frame and centre crop;
* External datasets 1 and 2 (as in Table 18).

Thresholds are fixed at 95% sensitivity on our test split, as for every student in Tables 17–18.
Models are saved to `baseline_tuned_models/`; the run is resumable. Expected GPU time ~30–40 min.

Output: `baseline_transfer.json` + `baseline_transfer_scores.npz` — upload both.

In [ ]:
# CELL 0 — CONFIG
DRIVE      = '/content/drive/MyDrive'
TEA_DRIVE  = f'{DRIVE}/Tea Leaf Dataset'                              # one folder per class
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
FEATS      = f'{RES}/features_dinov1_crop.npz'
SPLIT      = f'{RES}/_split.json'
ESCORES    = f'{RES}/E_scores_5rep.npz'          # published Experiment E, draws 0-4
ESCORES2   = f'{RES}/E_scores_rep5_9.npz'        # Experiment E, draws 5-9 (more_seeds.ipynb)
LOCAL      = '/content/tea_leaf_local'
CACHE      = '/content/crop_cache'
SEED, N_SHOT, CROP_PAD, IMG = 42, 10, 0.06, 224
REPS       = list(range(10))                     # the same ten label draws as Table 12
TUNED_NPZ  = f'{RES}/baseline_tuned_scores.npz'      # from baseline_tuned.ipynb: our-test scores to reproduce
OUT_NPZ    = f'{RES}/baseline_transfer_scores.npz'
OUT_JSON   = f'{RES}/baseline_transfer.json'
WDIR       = f'{RES}/baseline_tuned_models'          # this time the fine-tuned models are kept
# ---- field and external data (as in more_seeds.ipynb) ----
FIELD_ZIP_OR_DIR = f'{DRIVE}/Tea Field Dataset'
FIELD_FEAT_WHOLE = f'{RES}/field_test/feat_Field_whole.npz'
OUTF       = f'{RES}/experiment_F'
CENTRE_FRAC, DUP, TARGET_SENS = 0.60, 0.95, 0.95
EWU_DIR    = f'{DRIVE}/East West University dataset'
ALLY_DIR   = f'{DRIVE}/Allynugger'
# ---- pre-declared baselines (fixed before any result is seen; ALL are reported, none is selected on the test set) ----
FT_CONFIGS = {                                    # fine-tune MobileNetV3-Small (ImageNet init) on the 70 labelled leaves only
    'ft_lr1e-4_strongaug': dict(lr=1e-4, epochs=150, bs=16, aug='strong'),
    'ft_lr3e-4_strongaug': dict(lr=3e-4, epochs=150, bs=16, aug='strong'),
}
PROBE_BACKBONES = ['mobilenet_v3_small', 'resnet50']   # frozen ImageNet features + the same 70-label logistic probe as the DINO probe


In [ ]:
# CELL 1 — setup and local copy of the dataset
from google.colab import drive; drive.mount('/content/drive')
!pip -q install onnx onnxruntime timm
import os, io, re, json, time, glob, shutil, hashlib, random, warnings
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torchvision.models as tvm, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import ndimage
from scipy.stats import spearmanr
import cv2
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
os.makedirs(CACHE, exist_ok=True); os.makedirs(WDIR, exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png')

def _cp(t):
    s, d = t
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(s, d)
t0 = time.time()
tasks = [(os.path.join(r, f), os.path.join(LOCAL, os.path.relpath(r, TEA_DRIVE), f))
         for r, _, fs in os.walk(TEA_DRIVE) for f in fs if f.lower().endswith(IMG_EXT)]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('local copy: %d files in %.0fs' % (len(tasks), time.time() - t0))


In [ ]:
# CELL 2 — records in the original order: classes alphabetical, files sorted within class
classes = sorted(d for d in os.listdir(LOCAL) if os.path.isdir(os.path.join(LOCAL, d)))
HEALTHY = classes.index(next(c for c in classes if 'healthy' in c.lower()))
RECS = [dict(rel=f'{c}/{f}', path=os.path.join(LOCAL, c, f), y=k, d=int(k != HEALTHY))
        for k, c in enumerate(classes) for f in sorted(os.listdir(os.path.join(LOCAL, c)))
        if f.lower().endswith(IMG_EXT)]
print('classes:', classes, '| healthy index', HEALTHY)
print('records:', len(RECS))
assert len(classes) == 7 and len(RECS) == 13084, 'dataset differs from the published release'


In [ ]:
# CELL 3 — restore the group-aware split from _split.json, check labels against the cached features
SPL = json.load(open(SPLIT)); F = np.load(FEATS)
test_rel = ['/'.join(p.split('/')[-2:]) for p in SPL['test_paths']]
by_rel = {r['rel']: r for r in RECS}
missing = [p for p in test_rel if p not in by_rel]
assert not missing, f'{len(missing)} test files not found, e.g. {missing[:3]}'
TEST = [by_rel[p] for p in test_rel]                       # exact published test order
tset = set(test_rel)
TRAIN = [r for r in RECS if r['rel'] not in tset]          # record order (original convention)
ytr, dtr = np.array([r['y'] for r in TRAIN]), np.array([r['d'] for r in TRAIN])
yte, dte = np.array([r['y'] for r in TEST]), np.array([r['d'] for r in TEST])
print('train %d | test %d | test diseased %.4f' % (len(TRAIN), len(TEST), dte.mean()))
assert (len(TRAIN), len(TEST)) == (SPL['n_train'], SPL['n_test'])
assert (yte == F['yte']).all() and (dte == F['dte']).all(), 'test labels differ from cached features'
TRAIN_ORDER_OK = len(ytr) == len(F['ytr']) and (ytr == F['ytr']).all()
print('train label sequence matches cache:', TRAIN_ORDER_OK, '(order within class is checked in Cell 6)')


In [ ]:
# CELL 4 — segmentation and leaf crop
# Prefer the ORIGINAL modules from Drive (shape_confound.leaf_mask, feature_shape_control.crop_to_leaf);
# fall back to the paper's recipe: Otsu on HSV V|S, largest component, fill holes, bbox + 6% pad, 224x224.
import sys
for m in ('shape_confound.py', 'feature_shape_control.py'):
    hit = glob.glob(f'{STUDY}/**/{m}', recursive=True)
    if hit: shutil.copy(hit[0], f'/content/{m}')
sys.path.insert(0, '/content')
try:
    import shape_confound as _sc, feature_shape_control as _fsc
    _ORIG = True
except Exception as e:
    _ORIG = False; print('original modules not found (%s) -> fallback recipe' % e)

def _fb_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0); lab, n = ndimage.label(m)
    if n == 0: return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def _fb_crop(rgb, m, pad=CROP_PAD, size=IMG):
    ys, xs = np.where(m); H, W = m.shape
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0, x1, y1 = int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))
    return Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)

def leaf_crop(path):
    rgb = np.array(Image.open(path).convert('RGB'))
    if _ORIG:
        try:
            try: m = _sc.leaf_mask(rgb)
            except Exception: m = _sc.leaf_mask(Image.fromarray(rgb))
            im = _fsc.crop_to_leaf(rgb, np.asarray(m).astype(bool))
            im = im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im).astype(np.uint8))
            return im.convert('RGB').resize((IMG, IMG)) if im.size != (IMG, IMG) else im.convert('RGB')
        except Exception:
            pass
    return _fb_crop(rgb, _fb_mask(rgb))
print('crop function:', 'ORIGINAL modules' if _ORIG else 'fallback recipe')


In [ ]:
# CELL 5 — crop cache (JPEG q95, same as the revision_v1 run), parallel
def cache_path(rec):
    return os.path.join(CACHE, hashlib.md5(rec['rel'].encode()).hexdigest() + '.jpg')
def _build(rec):
    cp = cache_path(rec)
    if not os.path.exists(cp):
        try: leaf_crop(rec['path']).save(cp, quality=95)
        except Exception as e: return rec['rel'], str(e)
    return None
t0 = time.time()
with ProcessPoolExecutor(os.cpu_count()) as ex:
    fails = [f for f in ex.map(_build, RECS, chunksize=32) if f]
print('crop cache: %d images, %d failures, %.0fs' % (len(RECS), len(fails), time.time() - t0))
assert not fails, fails[:5]


In [ ]:
# CELL 6 — tuned baselines on the 70 labelled leaves (binary labels, as the published labels-only baseline)
import torch.nn.functional as Fnn
TO_T = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
AUG = {'strong': T.Compose([T.RandomResizedCrop(IMG, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                            T.RandomRotation(30), T.ColorJitter(.3, .3, .3, .05)])}
class DS(Dataset):
    def __init__(self, recs, labels, aug=None): self.r, self.l, self.aug = recs, labels, aug
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        im = Image.open(cache_path(self.r[i])).convert('RGB')
        if self.aug is not None: im = self.aug(im)
        return TO_T(im), int(self.l[i])
def draw(rep, n=N_SHOT):
    r = np.random.default_rng(SEED * 1000 + rep)
    return np.concatenate([r.choice(np.where(ytr == c)[0], n, replace=False) for c in range(len(classes))])
def finetune(recs, labels, seed, lr, epochs, bs, aug):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2); m = m.to(device)
    dl = DataLoader(DS(recs, labels, AUG[aug]), batch_size=bs, shuffle=True, num_workers=4, drop_last=False)
    opt = optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(dl)); lf = nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train()
        for x, y in dl:
            opt.zero_grad(); lf(m(x.to(device)), y.to(device)).backward(); opt.step(); sch.step()
    return m.eval()
@torch.no_grad()
def scores(m, recs):
    dl = DataLoader(DS(recs, np.zeros(len(recs))), batch_size=256, num_workers=4)
    return np.concatenate([torch.softmax(m(x.to(device)), 1)[:, 1].cpu().numpy() for x, _ in dl])
# the label draws must be the published ones: refit the DINO probe from the cached features and compare with Table 12
_pa = [roc_auc_score(dte, 1 - LogisticRegression(C=1.0, max_iter=2000).fit(F['Xtr'][draw(r)], ytr[draw(r)]).predict_proba(F['Xte'])[:, HEALTHY]) for r in range(5)]
assert TRAIN_ORDER_OK and np.allclose(_pa, [0.9791, 0.9858, 0.9888, 0.9836, 0.9893], atol=1e-3), ('label draws differ from the published run', _pa)
print('label draws reproduce the published probes:', np.round(_pa, 4))


In [ ]:
# CELL F1 — field records (same order as field_test.ipynb), whole-frame and centre crops into the cache
import zipfile
from collections import Counter
FIELD_LOCAL = '/content/field_local'
if not os.path.exists(FIELD_LOCAL):
    zips = glob.glob(f'{FIELD_ZIP_OR_DIR}/**/*.zip', recursive=True)
    if zips:
        for z in zips: zipfile.ZipFile(z).extractall(FIELD_LOCAL)
    else: shutil.copytree(FIELD_ZIP_OR_DIR, FIELD_LOCAL)
    for z in glob.glob(f'{FIELD_LOCAL}/**/*.zip', recursive=True): zipfile.ZipFile(z).extractall(os.path.dirname(z))
def field_class(p):
    d = os.path.basename(os.path.dirname(p)).lower().replace('_', ' ')
    return ('Healthy' if 'healthy' in d else 'Red Rust' if 'rust' in d else 'Helopeltis' if 'helopelt' in d
            else 'Blight' if 'blight' in d else None)
files = sorted(p for p in glob.glob(f'{FIELD_LOCAL}/**/*', recursive=True)
               if p.lower().endswith(IMG_EXT) and '__MACOSX' not in p and field_class(p))
FREC = []
for p in files:
    c = field_class(p); rel = f'FIELD/{c}/{os.path.basename(p)}'
    FREC.append(dict(path=p, cls=c, d=int(c != 'Healthy'),
                     whole=dict(rel=rel + '#whole'), centre=dict(rel=rel + '#centre')))
fy = np.array([r['d'] for r in FREC]); fcls = np.array([r['cls'] for r in FREC])
W = np.load(FIELD_FEAT_WHOLE, allow_pickle=True)
assert len(fy) == len(W['y']) and (fy == W['y']).all() and (fcls == W['cls']).all(), \
    'field records do not align with feat_Field_whole.npz (same download/order as field_test.ipynb needed)'
Xf_whole = W['X']
print('field:', Counter(fcls), '| aligned with field_test features')
def _fcrops(r):
    rgb = np.array(Image.open(r['path']).convert('RGB')); H, Wd = rgb.shape[:2]
    out = {}
    for key, box in (('whole', (0, 0, Wd, H)),
                     ('centre', (Wd * (1 - CENTRE_FRAC) / 2, H * (1 - CENTRE_FRAC) / 2,
                                 Wd * (1 + CENTRE_FRAC) / 2, H * (1 + CENTRE_FRAC) / 2))):
        cp = cache_path(r[key])
        if not os.path.exists(cp):
            x0, y0, x1, y1 = map(int, box)
            Image.fromarray(rgb[y0:y1, x0:x1]).resize((IMG, IMG), Image.BILINEAR).save(cp, quality=95)
    return True
t0 = time.time()
with ThreadPoolExecutor(8) as ex: list(ex.map(_fcrops, FREC))
print('field crops cached (%.0fs)' % (time.time() - t0))


In [ ]:
# CELL F3 — field train/test split: near-duplicate groups kept together, stratified by class
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
grp = connected_components(csr_matrix(Xf_whole @ Xf_whole.T >= DUP), directed=False)[1]
rng = np.random.default_rng(2026)
is_tr = np.zeros(len(FREC), bool)
for c in np.unique(fcls):
    gs = np.unique(grp[fcls == c]); rng.shuffle(gs)
    cum, half = 0, (fcls == c).sum() / 2
    for g in gs:
        if cum >= half: break
        is_tr[grp == g] = True; cum += (grp == g).sum()
assert not set(grp[is_tr]) & set(grp[~is_tr]), 'a near-duplicate group crosses the split'
F_TR, F_TE = np.where(is_tr)[0], np.where(~is_tr)[0]
print('field-train %d | field-test %d | groups crossing the split: 0' % (len(F_TR), len(F_TE)))
print('field-test classes:', Counter(fcls[F_TE]))
_sp = f'{OUTF}/field_split.json'
if os.path.exists(_sp):
    _ref = json.load(open(_sp)); _te = [FREC[i]['whole']['rel'][:-6] for i in F_TE]
    assert sorted(_te) == sorted(_ref['test']), 'field held-out half differs from Table 17'
    print('field held-out half matches Table 17 (%d photographs)' % len(_te))


In [ ]:
# CELL B1 — (verbatim from external_test.ipynb) preprocessing (same recipe as the paper: Otsu on HSV V|S, largest component,
#           fill holes, bbox crop with 6% padding, resize 224x224, JPEG q95 round-trip)
def leaf_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0)
    lab, n = ndimage.label(m)
    if n == 0:
        return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def box_of(mask):
    ys, xs = np.where(mask)
    return xs.min(), ys.min(), xs.max() + 1, ys.max() + 1

def crop_box(rgb, box, pad=0.06, size=IMG):
    H, W = rgb.shape[:2]; x0, y0, x1, y1 = box
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0 = int(max(0, x0 - px)), int(max(0, y0 - py))
    x1, y1 = int(min(W, x1 + px)), int(min(H, y1 + py))
    im = Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=95)       # match the cached-crop pipeline
    return Image.open(io.BytesIO(buf.getvalue())).convert('RGB')

def load_rgb(p):
    return np.array(Image.open(p).convert('RGB'))

def otsu_crop(rgb):
    m = leaf_mask(rgb)
    return crop_box(rgb, box_of(m)), float(m.mean())

def whole_crop(rgb):
    H, W = rgb.shape[:2]
    return crop_box(rgb, (0, 0, W, H), pad=0.0), 1.0


In [ ]:
!pip -q install pyyaml
import yaml, zipfile
# CELL B2 — (verbatim from external_test.ipynb) EWU: unzip Roboflow export, read YOLO boxes, drop Roboflow augmented copies
EWU_X = '/content/ewu'
if not os.path.exists(EWU_X):
    z = [p for p in glob.glob(f'{EWU_DIR}/*.zip')]
    assert z, 'no zip found in ' + EWU_DIR
    zipfile.ZipFile(z[0]).extractall(EWU_X)
yml = glob.glob(f'{EWU_X}/**/data.yaml', recursive=True) or glob.glob(f'{EWU_DIR}/data.yaml')
names = yaml.safe_load(open(yml[0]))['names']
names = list(names.values()) if isinstance(names, dict) else list(names)
print('EWU classes:', names)

def rf_stem(fname):          # Roboflow: <orig>_jpg.rf.<hash>.jpg  -> <orig>
    m = re.match(r'^(.*?)_(jpe?g|png|bmp|webp)\.rf\.', os.path.basename(fname), re.I)
    return m.group(1) if m else os.path.splitext(os.path.basename(fname))[0]

def split_rank(p):           # prefer un-augmented splits when de-duplicating
    return 0 if '/test/' in p else 1 if '/valid/' in p else 2

imgs = sorted([p for p in glob.glob(f'{EWU_X}/**/*', recursive=True) if p.lower().endswith(IMG_EXT)],
              key=lambda p: (split_rank(p), p))   # deterministic: same copy kept every run
seen, EWU = set(), []
for p in imgs:
    st = rf_stem(p)
    if st in seen: continue
    lbl = re.sub(r'/images/', '/labels/', p); lbl = os.path.splitext(lbl)[0] + '.txt'
    if not os.path.exists(lbl): continue
    seen.add(st)
    for line in open(lbl).read().strip().splitlines():
        v = line.split()
        if len(v) != 5: continue          # skip polygon rows
        EWU.append(dict(path=p, cls=names[int(v[0])], yolo=tuple(map(float, v[1:]))))
print('EWU: %d unique source images, %d leaf boxes (of %d image files incl. augmentations)'
      % (len(seen), len(EWU), len(imgs)))
from collections import Counter; print(Counter(r['cls'] for r in EWU))

def yolo_to_box(yolo, W, H):
    cx, cy, w, h = yolo
    return (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H


In [ ]:
# CELL B3 — (verbatim from external_test.ipynb) Allynugger: one folder per class, copied to local disk first (Drive reads are slow)
ALLY_LOCAL = '/content/ally_local'
src_files = [p for p in sorted(glob.glob(f'{ALLY_DIR}/*/*')) if p.lower().endswith(IMG_EXT)]
def _cp(p):
    d = os.path.join(ALLY_LOCAL, *p.split('/')[-2:])
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(p, d)
    return d
t0 = time.time()
with ThreadPoolExecutor(16) as ex: local_files = list(ex.map(_cp, src_files))
ALLY = [dict(path=p, cls=os.path.basename(os.path.dirname(p))) for p in local_files]
print('Allynugger: %s  (copied in %.0fs)' % (Counter(r['cls'] for r in ALLY), time.time() - t0))


In [ ]:
# CELL B4' — external crops (as in Table 18)
def _one(args):
    r, mode = args
    try:
        rgb = load_rgb(r['path']); H, W = rgb.shape[:2]
        im = crop_box(rgb, yolo_to_box(r['yolo'], W, H)) if mode == 'box' else whole_crop(rgb)[0]
        buf = io.BytesIO(); im.save(buf, 'JPEG', quality=95); return buf.getvalue()
    except Exception: return None
def build(records, mode):
    with ThreadPoolExecutor(8) as ex: res = list(ex.map(_one, [(r, mode) for r in records]))
    keep = [i for i, b in enumerate(res) if b is not None]
    return dict(crops=[Image.open(io.BytesIO(res[i])).convert('RGB') for i in keep],
                y=np.array([0 if 'healthy' in records[i]['cls'].lower() else 1 for i in keep]), cls=np.array([records[i]['cls'] for i in keep]))
XSETS = {'External_1': build(EWU, 'box'), 'External_2': build(ALLY, 'whole')}
for k, S in XSETS.items(): print(k, len(S['y']), 'healthy', (S['y'] == 0).sum())
assert len(XSETS['External_1']['y']) == 3007 and len(XSETS['External_2']['y']) == 883, 'external sets differ from Table 15'


In [ ]:
# CELL 9 — retrain the 20 tuned fine-tunes (same seeds), check our-test scores, score field and external data
from scipy.stats import wilcoxon
def thr_at_sens(s, d, t=TARGET_SENS): return float(np.sort(s[d == 1])[int(np.floor((1 - t) * (d == 1).sum()))])
def sens_spec(s, d, thr): f = s >= thr; return float(f[d == 1].mean()), float((~f)[d == 0].mean())
@torch.no_grad()
def pil_scores(m, pils, bs=128):
    return np.concatenate([torch.softmax(m(torch.stack([TO_T(p) for p in pils[i:i + bs]]).to(device)), 1)[:, 1].cpu().numpy()
                           for i in range(0, len(pils), bs)])
TUNED = np.load(TUNED_NPZ)
S = {k: v for k, v in np.load(OUT_NPZ).items()} if os.path.exists(OUT_NPZ) else {}
R = json.load(open(OUT_JSON))['per_model'] if os.path.exists(OUT_JSON) else {}
dF = fy[F_TE]
for name, c in FT_CONFIGS.items():
    for rep in REPS:
        key = f'{name}_rep{rep}'
        if key in R: print(key, 'done -> skipped'); continue
        t0 = time.time(); wp = f'{WDIR}/{key}.pt'
        if os.path.exists(wp):
            m = tvm.mobilenet_v3_small(weights=None); m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
            m.load_state_dict(torch.load(wp, map_location='cpu')); m = m.to(device).eval()
        else:
            idx = draw(rep); m = finetune([TRAIN[i] for i in idx], dtr[idx], seed=3000 + rep, **c); torch.save(m.state_dict(), wp)
        s_te = scores(m, TEST); a_te = roc_auc_score(dte, s_te); a_pub = roc_auc_score(dte, TUNED[key])
        thr = thr_at_sens(s_te, dte); row = dict(tealeaf_auc=float(a_te), tealeaf_auc_baseline_tuned_run=float(a_pub))
        for view in ('whole', 'centre'):
            sv_all = scores(m, [r[view] for r in FREC]); sv = sv_all[F_TE]; S[f'{key}_field_{view}'] = sv_all
            row[f'field_{view}'] = dict(auc_heldout=float(roc_auc_score(dF, sv)), auc_all2500=float(roc_auc_score(fy, sv_all)),
                                        sens_spec_fixed=sens_spec(sv, dF, thr))
        for tag, X in XSETS.items():
            sx = pil_scores(m, X['crops']); S[f'{key}_{tag}'] = sx
            row[tag] = dict(auc=float(roc_auc_score(X['y'], sx)), sens_spec_fixed=sens_spec(sx, X['y'], thr))
        S[f'{key}_tealeaf'] = s_te; R[key] = row
        np.savez(OUT_NPZ, **S); json.dump(dict(per_model=R), open(OUT_JSON, 'w'), indent=1)
        print('%-24s ours %.4f (first run %.4f) | field whole %.3f centre %.3f | Ext1 %.3f Ext2 %.3f | %.1f min'
              % (key, a_te, a_pub, row['field_whole']['auc_heldout'], row['field_centre']['auc_heldout'],
                 row['External_1']['auc'], row['External_2']['auc'], (time.time() - t0) / 60))
        del m; torch.cuda.empty_cache()


In [ ]:
# CELL 10 — summary (means over ten draws) and save
OUT = dict(per_model=R, summary={})
for name in FT_CONFIGS:
    rows = [R[f'{name}_rep{r}'] for r in REPS]
    g = lambda f: [float(np.mean([f(x) for x in rows])), float(np.std([f(x) for x in rows], ddof=1))]
    OUT['summary'][name] = dict(tealeaf=g(lambda x: x['tealeaf_auc']),
        field_whole_heldout=g(lambda x: x['field_whole']['auc_heldout']), field_centre_heldout=g(lambda x: x['field_centre']['auc_heldout']),
        field_whole_all=g(lambda x: x['field_whole']['auc_all2500']), field_centre_all=g(lambda x: x['field_centre']['auc_all2500']),
        field_whole_sens_spec=np.mean([x['field_whole']['sens_spec_fixed'] for x in rows], 0).tolist(),
        field_centre_sens_spec=np.mean([x['field_centre']['sens_spec_fixed'] for x in rows], 0).tolist(),
        External_1=g(lambda x: x['External_1']['auc']), External_2=g(lambda x: x['External_2']['auc']),
        External_1_sens_spec=np.mean([x['External_1']['sens_spec_fixed'] for x in rows], 0).tolist(),
        External_2_sens_spec=np.mean([x['External_2']['sens_spec_fixed'] for x in rows], 0).tolist(),
        max_abs_diff_vs_first_run=float(max(abs(x['tealeaf_auc'] - x['tealeaf_auc_baseline_tuned_run']) for x in rows)))
    s = OUT['summary'][name]
    print('%-22s ours %.3f | field whole %.3f ± %.3f, centre %.3f ± %.3f (held-out half) | Ext1 %.3f | Ext2 %.3f | rerun diff <= %.4f'
          % (name, s['tealeaf'][0], *s['field_whole_heldout'], *s['field_centre_heldout'], s['External_1'][0], s['External_2'][0], s['max_abs_diff_vs_first_run']))
json.dump(OUT, open(OUT_JSON, 'w'), indent=1); print('saved', OUT_JSON, 'and', OUT_NPZ, '-> upload both')
